## This demo showcases the implementation of user stories 404 and 582

This notebook shows the implementation of the different wrappers (rs-client-libraries) functions for the staging endpoints.

In [1]:
import requests
import os
import pprint
import time
import pystac
from pystac import Asset, Collection, Extent, Item, SpatialExtent, TemporalExtent, ItemCollection
# Init environment before running a demo notebook.
from resources.utils import *

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)
session = requests.Session()
auxip_client, cadip_client, catalog_client, staging_client = init_demo()

if os.getenv("RSPY_LOCAL_MODE") == "1":
    href_cadip = "http://rs-server-cadip:8000"
    href_adgs = "http://rs-server-adgs:8000"    
else:
    href_cadip = href_adgs = os.environ["RSPY_WEBSITE"]
    session.cookies.set("session", os.environ["RSPY_OAUTH2_COOKIE"])

cadip_collection_id = "cadip_sentinel1"
adgs_collection_id = "adgs"
TIMEOUT = 10
collection_description = Collection(
    id=TEST_COLLECTION,
    description=None,  # rs-client will provide a default description for us
    extent=Extent(
        spatial=SpatialExtent(bboxes=[-180.0, -90.0, 180.0, 90.0]),
        temporal=TemporalExtent([start_date, stop_date]),
    ),
)

# Init the dask cluster
from resources.dask_utils import *
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.dask_utils import *

# Use the staging cluster
dask_gateway = dask_gateway_staging
dask_cluster = dask_cluster_staging

DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.1"


Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
Connecting to dask gateway for 'dask-staging': http://dask-staging:8000 ...
Get existing dask cluster: '89c7479624374cadbf26fd60109f465b'
Dask dashboard for 'dask-staging': http://localhost:8701/clusters/89c7479624374cadbf26fd60109f465b/status
Dask workers for 'dask-staging' are up: 2/2


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


### Create a test collection in catalog named my_test_collection and check it afterwards to see if it's empty

In [2]:
# Create a test collection 
collection = create_test_collection()

# Check the catalog for my_test_collection
items = catalog_client.get_items(TEST_COLLECTION)
assert not list(items)

22:55:43.927 [INFO] (rs_client.rs_client) Retrieving all items from collection 'mcolinde:my_test_collection'.


### Check the available stac collections for stations (cadip, adgs) and for the stac catalog (the newly created one)

In [3]:
for client in [auxip_client, cadip_client, catalog_client]:
    collections = client.get_collections()
    print(f"\nCollections response:")
    for collection in collections:
        print(f"ID: {collection.id}, Title: {collection.title}")
    


Collections response:
ID: adgs, Title: All from 'adgs' station
ID: adgs2, Title: All from 'adgs2' station
ID: adgs_aux_pp2, Title: AUX_PP2 'adgs' station
ID: adgs_oper_aux_ecmwfd_pdmc, Title: OPER_AUX_ECMWFD_PDMC 'adgs' station
ID: adgs_oper_aux_obmemc, Title: OPER_AUX_OBMEMC 'adgs' station
ID: adgs_oper_aux_obmemc_pdmc, Title: OPER_AUX_OBMEMC_PDMC 'adgs' station
ID: adgs_oper_aux_preorb_opod, Title: OPER_AUX_PREORB_OPOD 'adgs' station
ID: adgs_oper_aux_resorb_opod, Title: OPER_AUX_RESORB_OPOD 'adgs' station
ID: adgs_oper_aux_resorb_opods, Title: OPER_AUX_RESORB_OPODs 'adgs' station
ID: adgs_oper_mpl_orbpre, Title: OPER_MPL_ORBPRE 'adgs' station
ID: adgs_oper_mpl_orbsct, Title: OPER_MPL_ORBSCT 'adgs' station
ID: adgs2_aux_pp2, Title: AUX_PP2 'adgs2' station
ID: adgs2_oper_aux_ecmwfd_pdmc, Title: OPER_AUX_ECMWFD_PDMC 'adgs2' station
ID: adgs2_oper_aux_obmemc, Title: OPER_AUX_OBMEMC 'adgs2' station
ID: adgs2_oper_aux_obmemc_pdmc, Title: OPER_AUX_OBMEMC_PDMC 'adgs2' station
ID: adgs2_ope

### Call the queryables (by using the rs-client) for .....

#### ADGS

In [4]:
general_queryables = auxip_client.get_queryables()
assert isinstance(general_queryables, dict)
pprint.PrettyPrinter(indent=4).pprint(general_queryables)
print("\n\n")
collection_queryables = auxip_client.get_collection_queryables("adgs_aux_pp2")
assert isinstance(collection_queryables, dict)
pprint.PrettyPrinter(indent=4).pprint(general_queryables)

{   '$id': 'http://rs-server-adgs:8000/auxip/queryables',
    '$schema': 'http://json-schema.org/draft-07/schema#',
    'properties': {   'Name': {   'description': 'string',
                                  'enum': None,
                                  'format': 'string',
                                  'pattern': None,
                                  'title': 'Name',
                                  'type': 'string'},
                      'auxip:id': {   'description': 'id',
                                      'enum': None,
                                      'format': 'string',
                                      'pattern': None,
                                      'title': 'auxip:id',
                                      'type': 'string'},
                      'constellation': {   'description': 'string',
                                           'enum': [   'sentinel-1',
                                                       'sentinel-2',
                      

#### CADIP

In [5]:
general_queryables = cadip_client.get_queryables()
assert isinstance(general_queryables, dict)
pprint.PrettyPrinter(indent=4).pprint(general_queryables)
print("\n\n")
collection_queryables = cadip_client.get_collection_queryables(cadip_collection_id)
assert isinstance(collection_queryables, dict)
pprint.PrettyPrinter(indent=4).pprint(general_queryables)

{   '$id': 'http://rs-server-cadip:8000/cadip/queryables',
    '$schema': 'http://json-schema.org/draft-07/schema#',
    'properties': {   'cadip:acquisition_id': {   'description': 'AcquisitionId',
                                                  'enum': None,
                                                  'format': 'string',
                                                  'pattern': None,
                                                  'title': 'cadip:acquisition_id',
                                                  'type': 'string'},
                      'cadip:antenna_id': {   'description': 'AntennaId',
                                              'enum': None,
                                              'format': 'string',
                                              'pattern': None,
                                              'title': 'cadip:antenna_id',
                                              'type': 'string'},
                      'cadip:antenna_status_o

#### STAC Catalog

In [6]:
general_queryables = catalog_client.get_queryables()
assert isinstance(general_queryables, dict)
pprint.PrettyPrinter(indent=4).pprint(general_queryables)
print("\n\n")
collection_queryables = catalog_client.get_collection_queryables(TEST_COLLECTION)
assert isinstance(collection_queryables, dict)
pprint.PrettyPrinter(indent=4).pprint(general_queryables)

{   '$id': 'http://rs-server-catalog:8000/queryables',
    '$schema': 'http://json-schema.org/draft-07/schema#',
    'additionalProperties': True,
    'properties': {   'datetime': {   'description': 'Datetime',
                                      'format': 'date-time',
                                      'pattern': '(\\+00:00|Z)$',
                                      'title': 'Acquired',
                                      'type': 'string'},
                      'eo:snow_cover': {'type': 'string'},
                      'expires': {'type': 'string'},
                      'geometry': {   '$ref': 'https://geojson.org/schema/Feature.json',
                                      'description': 'Item Geometry',
                                      'title': 'Item Geometry'},
                      'id': {   '$ref': 'https://schemas.stacspec.org/v1.0.0/item-spec/json-schema/item.json#/definitions/core/allOf/2/properties/id',
                                'description': 'Item ident

### Get all the items from the collection "cadip_sentinel1" found in the configuration of the CADIP station
These items are in fact sessions in case of a cadip station
"get_items(<collection_name>) function should retrieve all the items from that collections


In [7]:
items_collection_cadip = list(cadip_client.get_items(cadip_collection_id))
assert len(items_collection_cadip) > 0
for item in items_collection_cadip:
    print(f"Session {item.id} has {len(item.assets)} assets with datetime {item.properties.get('datetime')}")

22:55:44.268 [INFO] (rs_client.rs_client) Retrieving all items from collection 'cadip_sentinel1'.


Session S1A_20231120061537234567 has 60 assets with datetime 2023-11-20T06:15:37.234Z
Session S1A_20220715090550123456 has 40 assets with datetime 2022-07-15T09:05:50.123Z
Session S1A_20210410031928012345 has 21 assets with datetime 2021-04-10T03:39:28.012Z
Session S1A_20200105072204051312 has 60 assets with datetime 2020-01-05T18:52:26.165Z


### Request 14 items from the collection "adgs" found in the configuration of the ADGS station
By using the "search" function with a max_items set to 14, we are going to limit the result

In [8]:
# Another example on how to get the items, will just load all the results in memory
# items_collection_adgs = list(auxip_client.get_items(adgs_collection_id))
items_collection_adgs = auxip_client.search(max_items = 14, collections = [adgs_collection_id])
assert len(items_collection_adgs) == 14

#pprint.PrettyPrinter(indent=4).pprint(items_collection_adgs)
for item in items_collection_adgs:
    print(f"AUXIP asset {item.id} has datetime {item.properties.get('datetime')}")

AUXIP asset S1A_OPER_AUX_PREORB_OPOD_20240527T062732_V20240527T062732_20240527T062732.EOF has datetime 2024-05-27T09:44:09.509Z
AUXIP asset S1A_OPER_MPL_ORBSCT_20240514T150704_99999999T999999_0025.EOF has datetime 2024-05-14T09:10:52.206Z
AUXIP asset S1A_OPER_AUX_RESORB_OPOD_20240214T110702_V20240214T071044_20240214T102814.EOF has datetime 2024-02-14T02:08:13.372Z
AUXIP asset S1A_OPER_AUX_RESORB_OPOD_20240204T110702_V20240204T071044_20240204T102814.EOF has datetime 2024-02-04T07:58:06.866Z
AUXIP asset S1A_OPER_AUX_RESORB_OPOD_20240129T110702_V20240129T071044_20240129T102814.EOF has datetime 2024-01-29T01:56:47.075Z
AUXIP asset S1A_OPER_MPL_ORBSCT_20240115T150704_99999999T999999_0025.EOF has datetime 2024-01-15T15:40:00.653Z
AUXIP asset S1A_OPER_AUX_OBMEMC_PDMC_20240106T000000.xml has datetime 2024-01-06T07:20:58.166Z
AUXIP asset S1A_OPER_AUX_RESORB_OPOD_20231218T110702_V20231218T071044_20231218T102814.EOF has datetime 2023-12-18T23:27:56.465Z
AUXIP asset S1A_OPER_AUX_PREORB_OPOD_202310

### Call the search method by using a cql2-text filter

In [9]:
auxip_client.search(method = "GET", stac_filter="processing:facility='FOS' AND product:type='AUX_PP2'")

### Check existing processes + display information about the staging process

In [10]:
# Returns list of all available processes from config.
processes = staging_client.get_processes()
print(processes)

{'processes': [{'id': 'Staging', 'version': '1.0.0'}], 'links': [{'href': 'http://rs-server-staging:8000/processes', 'rel': 'self', 'type': 'application/json', 'title': 'List of processes'}]}


In [11]:
# Should return info about the staging process.
process_info = staging_client.get_process("staging")
print(process_info)

{'id': 'Staging', 'version': '1.0.0'}


### Check the jobs table

In [12]:
jobs = staging_client.get_jobs()
if jobs.get("numberMatched") > 0:
    delete_jobs = True
    if cluster_mode == True:
        delete_jobs = input(f"There are {jobs.get('numberMatched')} jobs in the table. Do you want to delete them all (y/n)?").lower().strip() == 'y'
    if delete_jobs:
        print("Deleting all the jobs...")
        for job in jobs.get("jobs"):
            delete_response = staging_client.delete_job(job.get("jobID"))
        # Check that the jobs have been deleted
        jobs = staging_client.get_jobs()
        print(f"Existing jobs: {jobs}")

Deleting all the jobs...
Existing jobs: {'jobs': [], 'links': [{'href': 'string', 'rel': 'service', 'type': 'application/json', 'hreflang': 'en', 'title': 'List of jobs'}], 'numberMatched': 0}


### Starting 2 staging processes, one from the CADIP station and one from the ADGS station
The staging process from the ADGS station is expected to fail because one of the assets contains an incorrect download link for the file.

In [13]:
staging_resp_list = []
for items in [pystac.ItemCollection(list(items_collection_cadip)), pystac.ItemCollection(list(items_collection_adgs))]:
    staging_resp_list.append(staging_client.run_staging(items.to_dict(), TEST_COLLECTION))
    
timeout = 120
started_job_id_list = []

for resp in staging_resp_list:
    started_job_id_list.append(resp["jobID"])
    while timeout > 0:
        if "running" not in resp["status"]:
            break
        # TODO: to replace with the following commented line after the rs-server-staging update
        ###job_info = staging_client.get_job_info(resp["jobID"])
        job_info = staging_client.get_job_info(resp["jobID"])
        pprint.PrettyPrinter(indent=4).pprint(job_info)
        print("\n")
        if "successful" in job_info["status"]:
            print(" ----- Job COMPLETED \n")
            break
        if "failed" in job_info["status"]:
            print("-----Job FAILED \n")
            break
        time.sleep(2)
        timeout -= 2

{   'created': datetime.datetime(2025, 3, 25, 22, 55, 49, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>),
    'jobID': 'd4448bc1-994a-44f1-baf0-efb916d2d96b',
    'message': 'In progress',
    'processID': 'staging',
    'progress': 3,
    'started': datetime.datetime(2025, 3, 25, 22, 55, 49, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>),
    'status': 'running',
    'type': 'process',
    'updated': datetime.datetime(2025, 3, 25, 22, 55, 52, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>)}


{   'created': datetime.datetime(2025, 3, 25, 22, 55, 49, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>),
    'jobID': 'd4448bc1-994a-44f1-baf0-efb916d2d96b',
    'message': 'In progress',
    'processID': 'staging',
    'progress': 7,
    'started': datetime.datetime(2025, 3, 25, 22, 55, 49, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>),
    'status': 'running',
    'type': 'process',
    'updated': datetime.datetime(2025, 3, 25, 22, 55, 54, tzinfo=<isodate.tzi

### Check the catalog collection "my_test_collection" for all the items:

In [14]:
# Check the catalog for my_test_collection
result = list(catalog_client.get_items(TEST_COLLECTION))

for item in result:
    print(f"Item {item.id} has {len(item.assets)} assets")


22:56:12.478 [INFO] (rs_client.rs_client) Retrieving all items from collection 'mcolinde:my_test_collection'.


Item S1A_OPER_AUX_PREORB_OPOD_20240527T062732_V20240527T062732_20240527T062732.EOF has 1 assets
Item S1A_OPER_MPL_ORBSCT_20240514T150704_99999999T999999_0025.EOF has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20240214T110702_V20240214T071044_20240214T102814.EOF has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20240204T110702_V20240204T071044_20240204T102814.EOF has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20240129T110702_V20240129T071044_20240129T102814.EOF has 1 assets
Item S1A_OPER_MPL_ORBSCT_20240115T150704_99999999T999999_0025.EOF has 1 assets
Item S1A_OPER_AUX_OBMEMC_PDMC_20240106T000000.xml has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20231218T110702_V20231218T071044_20231218T102814.EOF has 1 assets
Item S1A_20231120061537234567 has 60 assets
Item S1A_OPER_AUX_PREORB_OPOD_20231013T062732_V20231013T062732_20231013T062732.EOF has 1 assets
Item S1A_OPER_AUX_PREORB_OPOD_20231007T062732_V20231007T062732_20231007T062732.EOF has 1 assets
Item S1A_AUX_PP2_V20230818T080000_G20230818T080000.SAFE has 1 a

### Check the jobs table

In [15]:
# Check that each of the job previously launched are successful
for job_id in started_job_id_list:
    job_results = staging_client.get_job_results(job_id)
    print(f"Results from job {job_id}: {job_results}")
    assert job_results == "successful"

Results from job d4448bc1-994a-44f1-baf0-efb916d2d96b: successful
Results from job c73f7193-9005-4fb3-8574-cbc6c6da4a76: successful


### Delete the catalog collection and recreate it

In [16]:
# Create a test collection 
collection = create_test_collection()
# Check the catalog for my_test_collection
items = catalog_client.get_items(TEST_COLLECTION)
assert not list(items)

22:56:16.340 [INFO] (rs_client.rs_client) Retrieving all items from collection 'mcolinde:my_test_collection'.


### Stage some files from both cadip and auxip stations

In [17]:
items_cadip = stage_test_objects(cadip_client, 18, objects_are_files = True)
assert items_cadip
assert len(items_cadip.items[0].get_assets()) == 18
items_auxip = stage_test_objects(auxip_client, 14, objects_are_files = True)
assert items_auxip
assert len(items_auxip.items) == 14

{   'created': datetime.datetime(2025, 3, 25, 22, 56, 18, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>),
    'jobID': 'e517c7d9-ca49-48bb-8221-2e3a3bcfb8a9',
    'message': 'Sending tasks to the dask cluster',
    'processID': 'staging',
    'progress': 0,
    'started': datetime.datetime(2025, 3, 25, 22, 56, 18, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>),
    'status': 'running',
    'type': 'process',
    'updated': datetime.datetime(2025, 3, 25, 22, 56, 18, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>)}


{   'created': datetime.datetime(2025, 3, 25, 22, 56, 18, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>),
    'jobID': 'e517c7d9-ca49-48bb-8221-2e3a3bcfb8a9',
    'message': 'In progress',
    'processID': 'staging',
    'progress': 100,
    'started': datetime.datetime(2025, 3, 25, 22, 56, 18, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>),
    'status': 'running',
    'type': 'process',
    'updated': datetime.datetime(2025, 3, 25, 22, 56,

22:56:23.795 [INFO] (rs_client.rs_client) Retrieving specific items from collection 'mcolinde:my_test_collection'.


{   'created': datetime.datetime(2025, 3, 25, 22, 56, 26, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>),
    'jobID': '004872a4-10b2-4b1e-96b6-107e6fe4662c',
    'message': 'Sending tasks to the dask cluster',
    'processID': 'staging',
    'progress': 0,
    'started': datetime.datetime(2025, 3, 25, 22, 56, 26, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>),
    'status': 'running',
    'type': 'process',
    'updated': datetime.datetime(2025, 3, 25, 22, 56, 26, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>)}


{   'created': datetime.datetime(2025, 3, 25, 22, 56, 26, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>),
    'jobID': '004872a4-10b2-4b1e-96b6-107e6fe4662c',
    'message': 'In progress',
    'processID': 'staging',
    'progress': 100,
    'started': datetime.datetime(2025, 3, 25, 22, 56, 26, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>),
    'status': 'running',
    'type': 'process',
    'updated': datetime.datetime(2025, 3, 25, 22, 56,

22:56:32.061 [INFO] (rs_client.rs_client) Retrieving specific items from collection 'mcolinde:my_test_collection'.


### Check the catalog collection "my_test_collection" for all the items:

In [18]:
# Check the catalog for my_test_collection
result = list(catalog_client.get_items(TEST_COLLECTION))

for item in result:
    print(f"Item {item.id} has {len(item.assets)} assets")


22:56:32.201 [INFO] (rs_client.rs_client) Retrieving all items from collection 'mcolinde:my_test_collection'.


Item S1A_OPER_AUX_PREORB_OPOD_20240527T062732_V20240527T062732_20240527T062732.EOF has 1 assets
Item S1A_OPER_MPL_ORBSCT_20240514T150704_99999999T999999_0025.EOF has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20240214T110702_V20240214T071044_20240214T102814.EOF has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20240204T110702_V20240204T071044_20240204T102814.EOF has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20240129T110702_V20240129T071044_20240129T102814.EOF has 1 assets
Item S1A_OPER_MPL_ORBSCT_20240115T150704_99999999T999999_0025.EOF has 1 assets
Item S1A_OPER_AUX_OBMEMC_PDMC_20240106T000000.xml has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20231218T110702_V20231218T071044_20231218T102814.EOF has 1 assets
Item S1A_20231120061537234567 has 18 assets
Item S1A_OPER_AUX_PREORB_OPOD_20231013T062732_V20231013T062732_20231013T062732.EOF has 1 assets
Item S1A_OPER_AUX_PREORB_OPOD_20231007T062732_V20231007T062732_20231007T062732.EOF has 1 assets
Item S1A_AUX_PP2_V20230818T080000_G20230818T080000.SAFE has 1 a

### Example on how the user may stage all the items from the adgs collections, by using rs-client
First, delete the catalog collection and recreate it

In [19]:
# Create a test collection 
collection = create_test_collection()
# Check the catalog for my_test_collection
items = catalog_client.get_items(TEST_COLLECTION)
assert not list(items)

22:56:33.027 [INFO] (rs_client.rs_client) Retrieving all items from collection 'mcolinde:my_test_collection'.


### Stage all the items from the adgs collection that exists in the adgs station

In [20]:
# Get all the items from the adgs colection
items_iterator = auxip_client.get_items("adgs")
# Create a pystac.ItemCollection from the result
item_collection = pystac.ItemCollection(list(items_iterator))

item_collection.items = [
    item for item in item_collection.items
    if item.properties.get("product:type") != "AX___OSF_AX"
]
# Use the dictionary to start the staging
job = staging_client.run_staging(item_collection.to_dict(), TEST_COLLECTION)
while timeout > 0:
    if "running" not in job["status"]:
        break
    # TODO: to replace with the following commented line after the rs-server-staging update
    ###job_info = staging_client.get_job_info(resp["jobID"])
    job_info = staging_client.get_job_info(job["jobID"])
    pprint.PrettyPrinter(indent=4).pprint(job_info)
    print("\n")
    if "successful" in job_info["status"]:
        print(" ----- Job COMPLETED \n")
        break
    if "failed" in job_info["status"]:
        print("-----Job FAILED \n")
        break
    time.sleep(2)
    timeout -= 2

22:56:33.120 [INFO] (rs_client.rs_client) Retrieving all items from collection 'adgs'.


{   'created': datetime.datetime(2025, 3, 25, 22, 56, 35, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>),
    'jobID': '96d840a2-b05a-4ddc-8baa-f11f69d1ebbb',
    'message': 'Sending tasks to the dask cluster',
    'processID': 'staging',
    'progress': 0,
    'started': datetime.datetime(2025, 3, 25, 22, 56, 35, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>),
    'status': 'running',
    'type': 'process',
    'updated': datetime.datetime(2025, 3, 25, 22, 56, 36, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>)}


{   'created': datetime.datetime(2025, 3, 25, 22, 56, 35, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>),
    'jobID': '96d840a2-b05a-4ddc-8baa-f11f69d1ebbb',
    'message': 'In progress',
    'processID': 'staging',
    'progress': 12,
    'started': datetime.datetime(2025, 3, 25, 22, 56, 35, tzinfo=<isodate.tzinfo.Utc object at 0x7f0ad0db2290>),
    'status': 'running',
    'type': 'process',
    'updated': datetime.datetime(2025, 3, 25, 22, 56, 

### Check the catalog collection "my_test_collection" for all the items:

In [21]:
# Check the catalog for my_test_collection
result = list(catalog_client.get_items(TEST_COLLECTION))
assert len(result) == 93
for item in result:
    print(f"Item {item.id} has {len(item.assets)} assets")


22:57:00.953 [INFO] (rs_client.rs_client) Retrieving all items from collection 'mcolinde:my_test_collection'.


Item S1A_OPER_AUX_PREORB_OPOD_20240527T062732_V20240527T062732_20240527T062732.EOF has 1 assets
Item S1A_OPER_MPL_ORBSCT_20240514T150704_99999999T999999_0025.EOF has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20240214T110702_V20240214T071044_20240214T102814.EOF has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20240204T110702_V20240204T071044_20240204T102814.EOF has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20240129T110702_V20240129T071044_20240129T102814.EOF has 1 assets
Item S1A_OPER_MPL_ORBSCT_20240115T150704_99999999T999999_0025.EOF has 1 assets
Item S1A_OPER_AUX_OBMEMC_PDMC_20240106T000000.xml has 1 assets
Item S1A_OPER_AUX_RESORB_OPOD_20231218T110702_V20231218T071044_20231218T102814.EOF has 1 assets
Item S1A_OPER_AUX_PREORB_OPOD_20231013T062732_V20231013T062732_20231013T062732.EOF has 1 assets
Item S1A_OPER_AUX_PREORB_OPOD_20231007T062732_V20231007T062732_20231007T062732.EOF has 1 assets
Item S1A_AUX_PP2_V20230818T080000_G20230818T080000.SAFE has 1 assets
Item S1A_OPER_AUX_PREORB_OPOD_20230807

In [22]:
result = catalog_client.remove_collection(TEST_COLLECTION)
assert result.json()["deleted collection"] == TEST_COLLECTION
pp.pprint(result.json())

{'deleted collection': 'my_test_collection'}
